# 2-Factor Lifted CIR-HW — v35a (fine-tune of v35)

Minimal fine-tune of v35: same 8-node param space, same pure-vol-RMSE loss (no arb penalties — PAVA cleans up after), same long-end weighting, same coarse training grid. Only change: **warm-start from v35.pkl instead of smart_init, run 500 more iters at lr=3e-4 (0.3× v35's 1e-3)**.

Reference: v35 fine-grid RMSE = 0.013108. v35a target: anything below that.

Modest expected gain — v35 cosine-annealed lr down to 5e-5 over 2000 iters and was probably near its basin minimum. This buys a fresh Adam-momentum direction + different SPSA seed sequence; if there's a side exit in the basin we'll find it, otherwise we confirm v35 is the minimum.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CONFIG
# ═══════════════════════════════════════════════════════════════════════════
RETRAIN      = False
LOAD_VERSION = True
VERSION      = 'v35a'
RUN_TAG      = 'v35a'

REFERENCE_VERSION = 'v35'
REFERENCE_LOSS    = 0.013108   # v35 fine-timeline RMSE — the bar to beat

# ── Arb penalties OFF (same as v35) ────────────────────────────────────────
ENFORCE_BUTTERFLY_ARB = False
ENFORCE_CALENDAR_ARB  = False
LAMBDA_BUTTERFLY      = 1.0    # unused (penalties off)
BUTTERFLY_MODE        = 'strong'
LAMBDA_CALENDAR       = 3.0    # unused (penalties off)
LAMBDA_SOBOLEV        = 0.0

# ── Optimizer (FINE-TUNE: lower lr/c, fewer iters than v35) ────────────────
max_iter       = 500
patience       = 500
lr             = 3e-4   # 0.3× v35's 1e-3
c_spsa         = 1.5e-4  # 0.5× v35's 3e-4
n_paths_grad   = 16384
n_pert         = 8
seed_base      = 4242   # different from v35's 42 → independent SPSA sequence
N_RESTARTS_CFG = 1

PROTECTED_VERSIONS = {'v22', 'v22_best_1500iters_locked', 'v24', 'v25',
                      'v29a', 'v29c', 'v29i', 'v29d', 'v29e',
                      'v35', 'v37'}

USE_SMART_INIT  = False
WARM_START_FROM = 'v35'

print(f'VERSION={VERSION}  RETRAIN={RETRAIN}')
print(f'  Reference: {REFERENCE_VERSION} loss={REFERENCE_LOSS:.6f}')
print(f'  {max_iter} iter, lr={lr}, c={c_spsa}, restarts={N_RESTARTS_CFG}')
print(f'  warm-start from: {WARM_START_FROM}')
print(f'  arb penalties: OFF (PAVA post-processing handles arb-freeness)')

### Import key libraries and data

In [ ]:
import importlib
import pandas as pd
import numpy as np
import torch
from pathlib import Path

import sys
sys.path.insert(0, '../../..')
from pyquant.interest_rates import build_fwd_curve, build_ifwd_curve_from_now_starting

import affine_calibration.scripts.simulation as _sim
importlib.reload(_sim)
from affine_calibration.scripts.simulation import (
    fast_simulate_2factor, batch_price_caplets,
    theta_to_vec, rho_to_vec,
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

data_dir = Path('../../../../../data')
vol_key_rate = pd.read_csv(data_dir / 'volatility_key_rate.csv')
fwd_ois      = pd.read_csv(data_dir / 'forward_ois.csv')
fwd_key_rate = pd.read_csv(data_dir / 'forward_key_rate.csv')

### Forward curves

In [ ]:
key_fwd_spline = build_fwd_curve(
    torch.tensor(fwd_key_rate['forward_rate'].values, dtype=torch.float32),
    torch.tensor(fwd_key_rate['time_to_maturity'].values, dtype=torch.float32)
)
ois_fwd_spline = build_fwd_curve(
    torch.tensor(fwd_ois['forward_rate'].values, dtype=torch.float32),
    torch.tensor(fwd_ois['time_to_maturity'].values, dtype=torch.float32)
)
key_ifwd_spline = build_ifwd_curve_from_now_starting(
    torch.tensor(fwd_key_rate['forward_rate'].values, dtype=torch.float32),
    torch.tensor(fwd_key_rate['time_to_maturity'].values, dtype=torch.float32)
)
ois_ifwd_spline = build_ifwd_curve_from_now_starting(
    torch.tensor(fwd_ois['forward_rate'].values, dtype=torch.float32),
    torch.tensor(fwd_ois['time_to_maturity'].values, dtype=torch.float32)
)
print('Forward curves built.')

### Timeline (coarse for training, fine for eval — same as v35)

In [ ]:
T_MAX = 10.0

# Fine timeline (daily) — evaluation only
N_FINE = 3651
timeline = torch.linspace(0, T_MAX, N_FINE + 1, dtype=torch.float32, device=device)
dt = timeline[1] - timeline[0]

I_key = key_ifwd_spline.evaluate(timeline.cpu()).to(device)
I_ois = ois_ifwd_spline.evaluate(timeline.cpu()).to(device)
f_key_vec = torch.zeros_like(timeline)
f_key_vec[:-1] = torch.diff(I_key) / dt; f_key_vec[-1] = f_key_vec[-2]
f_ois_vec = torch.zeros_like(timeline)
f_ois_vec[:-1] = torch.diff(I_ois) / dt; f_ois_vec[-1] = f_ois_vec[-2]

# Coarse timeline (~3.5d/step) — training
N_COARSE = 1043
tl_coarse = torch.linspace(0, T_MAX, N_COARSE + 1, dtype=torch.float32, device=device)
dt_coarse = tl_coarse[1] - tl_coarse[0]

I_key_c = key_ifwd_spline.evaluate(tl_coarse.cpu()).to(device)
I_ois_c = ois_ifwd_spline.evaluate(tl_coarse.cpu()).to(device)
f_key_coarse = torch.zeros_like(tl_coarse)
f_key_coarse[:-1] = torch.diff(I_key_c) / dt_coarse; f_key_coarse[-1] = f_key_coarse[-2]
f_ois_coarse = torch.zeros_like(tl_coarse)
f_ois_coarse[:-1] = torch.diff(I_ois_c) / dt_coarse; f_ois_coarse[-1] = f_ois_coarse[-2]

r0 = f_ois_vec[0]
print(f'Fine   timeline: {N_FINE} steps, dt={dt:.6f}Y ({dt*365:.2f} days)')
print(f'Coarse timeline: {N_COARSE} steps, dt={dt_coarse:.6f}Y ({dt_coarse*365:.1f} days)')
print(f'r0 (instantaneous OIS) = {r0.item()*100:.2f}%')

### Caplet grid + market PVs/vegas

In [ ]:
from affine_calibration.scripts.caplet_vol_surface import compute_market_pvs, compute_market_vegas

caplet_data = vol_key_rate[['time_to_maturity', 'strike', 'implied_normal_vol']].values
T_fixes     = torch.tensor(caplet_data[:, 0], dtype=torch.float32, device=device)
strikes     = torch.tensor(caplet_data[:, 1], dtype=torch.float32, device=device)
market_vols = torch.tensor(caplet_data[:, 2], dtype=torch.float32, device=device)
tau = 0.25

idx_fixes   = torch.searchsorted(timeline, T_fixes).clamp(0, len(timeline) - 1)
idx_fixes_c = torch.searchsorted(tl_coarse, T_fixes).clamp(0, len(tl_coarse) - 1)

market_fwds = key_ifwd_spline.evaluate(T_fixes.cpu()).to(device) / T_fixes

market_pvs_cached = compute_market_pvs(T_fixes, strikes, market_vols, market_fwds, f_ois_vec, timeline)
market_pvs_coarse = compute_market_pvs(T_fixes, strikes, market_vols, market_fwds, f_ois_coarse, tl_coarse)

market_vegas_floored, n_vega_floored = compute_market_vegas(
    T_fixes, strikes, market_vols, market_fwds, f_ois_vec, timeline)

print(f'Caplets: {len(T_fixes)},  vegas floored: {n_vega_floored}/{len(T_fixes)}')

### Parameter space (v35a → 8 nodes, 27 params — same as v35)

In [ ]:
import affine_calibration.scripts.param_space as _ps
importlib.reload(_ps)
from affine_calibration.scripts.param_space import make_param_space

# param_space.py uses _version_number('v35a')=35 → 8 theta nodes, gamma upper=3.0 (same as v35)
ps = make_param_space(device, version=VERSION)

lb, ub          = ps['lb'], ps['ub']
param_range     = ps['param_range']
param_bounds    = ps['param_bounds']
param_names     = ps['param_names']
n_params        = ps['n_params']
theta_nodes     = ps['theta_nodes']
rho_nodes_t     = ps['rho_nodes_t']
pack            = ps['pack']
unpack          = ps['unpack']
warm_start_fn   = ps.get('warm_start')

calib_mask = torch.ones(len(strikes), dtype=torch.bool, device=device)
assert n_params == 27, f'expected 27 params (8 nodes), got {n_params}'
print(f'VERSION={VERSION}: {n_params} params, theta_nodes={theta_nodes.cpu().numpy().tolist()}')

### Loss factory (pure vol RMSE, long-end weighting — same as v35)

In [ ]:
# Maturity weights: long-end (≥7Y) upweighted 3× — same as v35 (inherited from v29i/v29e)
mats_unique = torch.unique(T_fixes).cpu().numpy()
raw_w = np.where(mats_unique >= 7.0, 3.0, 1.0).astype(np.float32)
raw_w /= raw_w.sum()
mat_weights = torch.tensor(raw_w, dtype=torch.float32, device=device)
print(f'Mat weights: short (<7Y)={raw_w[mats_unique < 7.0].mean():.4f},  long (≥7Y)={raw_w[mats_unique >= 7.0].mean():.4f}')

import affine_calibration.scripts.pricing_models as _pm;  importlib.reload(_pm)
import affine_calibration.scripts.calib_objective as co;  importlib.reload(co)
from affine_calibration.scripts.calib_objective import make_eval_loss

strike_weights = None  # same as v35

eval_loss = make_eval_loss(
    unpack, theta_nodes, rho_nodes_t, tl_coarse,
    f_key_coarse, f_ois_coarse, idx_fixes_c,
    strikes, market_pvs_coarse, market_vegas_floored,
    calib_mask, T_fixes, device,
    mat_weights=mat_weights,
    strike_weights=strike_weights,
    smooth_pen_weight=0.02,
    objective_mode='vol',
    market_vols=market_vols,
    sobolev_pen_weight=LAMBDA_SOBOLEV,
)
print('Loss: pure vol RMSE — all arb penalties disabled (PAVA handles arb-freeness post-hoc)')

### Fine-tune training (warm-start from v35, 500 iters at lr=3e-4)

In [ ]:
import affine_calibration.scripts.optimizers as _opt;     importlib.reload(_opt)
from affine_calibration.scripts.optimizers import run_adam_spsa

_ckpt = Path('calibration_weights') / f'fast_calibration_{RUN_TAG}.pkl'
if RETRAIN and _ckpt.exists() and RUN_TAG in PROTECTED_VERSIONS:
    raise RuntimeError(f"BLOCKED: '{_ckpt}' is protected.")

GRAD_CLIP = 10.0
print(f'Fine-tune v35a: warm from {WARM_START_FROM}, coarse grid (N={N_COARSE})')
print(f'  {max_iter} iter, lr={lr}, c={c_spsa}, seed_base={seed_base}')
print(f'  Autosave → {_ckpt.name}')

if RETRAIN:
    _ws_ckpt = Path('calibration_weights') / f'fast_calibration_{WARM_START_FROM}.pkl'
    if not _ws_ckpt.exists():
        raise FileNotFoundError(f"Warm-start checkpoint missing: {_ws_ckpt}")
    # Same 8-node space → param_space.warm_start works directly (no interpolation)
    p_init = warm_start_fn([(_ws_ckpt, WARM_START_FROM)])

    best_params, best_loss, history, model_pvs_k = run_adam_spsa(
        eval_loss, unpack, p_init,
        max_iter=max_iter, n_paths=n_paths_grad,
        lr=lr, c=c_spsa, seed_base=seed_base,
        param_names=param_names, param_bounds=param_bounds,
        param_range=param_range, n_params=n_params,
        theta_nodes=theta_nodes,
        autosave_path=_ckpt,
        market_pvs_cached=market_pvs_coarse,
        market_vegas_floored=market_vegas_floored,
        n_perturbations=n_pert,
        lr_schedule='cosine', lr_min_ratio=0.05,
        c_schedule='cosine',  c_min_ratio=0.2,
        grad_clip=GRAD_CLIP,
        patience=patience,
        n_restarts=N_RESTARTS_CFG, reset_momentum_on_restart=False,
        version=RUN_TAG,
        reference_loss=REFERENCE_LOSS,
        reference_version=REFERENCE_VERSION,
    )
    print(f'\n{"═"*70}')
    print(f'  DONE: best_loss={best_loss:.6f}  (reference {REFERENCE_VERSION}={REFERENCE_LOSS:.6f})')
    print(f'{"═"*70}')
else:
    print('RETRAIN=False — loading saved weights.')

In [ ]:
if not RETRAIN:
    print('Convergence plots available only after RETRAIN=True run.')
else:
    from affine_calibration.scripts.caplet_vol_surface import plot_spsa_convergence, params_to_dataframe
    plot_spsa_convergence(history, model_pvs=model_pvs_k, market_pvs=market_pvs_coarse)
    display(params_to_dataframe(best_params, theta_nodes))

### Evaluation — fine-timeline MC, compare against v35

In [ ]:
import affine_calibration.scripts.pricing_models as _pm;     importlib.reload(_pm)
import affine_calibration.scripts.caplet_vol_surface as _cvs; importlib.reload(_cvs)
import affine_calibration.scripts.calib_objective as co;     importlib.reload(co)
from affine_calibration.scripts.calib_objective import load_checkpoint, evaluate_vol_surface
from affine_calibration.scripts.caplet_vol_surface import params_to_dataframe

weights_dir = Path('calibration_weights')
load_version = RUN_TAG if RETRAIN else (globals().get('LOAD_VERSION') or RUN_TAG)
_load_path = weights_dir / f'fast_calibration_{load_version}.pkl'
if not _load_path.exists():
    raise FileNotFoundError(f"Checkpoint not found for version='{load_version}': {_load_path}")

best_params, theta_nodes, _ = load_checkpoint(_load_path, device)
print(f'Loaded for evaluation: {_load_path}')

VERSION = load_version
display(params_to_dataframe(best_params, theta_nodes))

vol_results, full_rmse, model_pvs_final = evaluate_vol_surface(
    best_params, theta_nodes, rho_nodes_t,
    timeline, f_key_vec, f_ois_vec,
    strikes, idx_fixes, T_fixes,
    vol_key_rate, fwd_key_rate, fwd_ois,
    device,
    csv_path=weights_dir / f'market_vs_model_{VERSION}.csv',
    version_name=VERSION,
)
print(f'Vol surface RMSE [{VERSION}]: {full_rmse:.6f}  (v35: {REFERENCE_LOSS:.6f})')
_delta = full_rmse - REFERENCE_LOSS
print(f'  Δ vs v35: {_delta:+.6f}  ({"WORSE" if _delta > 0 else "BETTER" if _delta < 0 else "same"})')

### 3D vol surface — market vs model

In [ ]:
import affine_calibration.scripts.caplet_vol_surface as cvs
importlib.reload(cvs)
from affine_calibration.scripts.caplet_vol_surface import plot_vol_surface_interactive

plot_vol_surface_interactive(vol_results, version_name=VERSION)

### Arbitrage check (Dupire calendar + butterfly, model surface, pre-PAVA)

In [ ]:
from affine_calibration.scripts.caplet_vol_surface import check_market_arbitrage

model_col = f'model_vol_{VERSION}'
assert model_col in vol_results.columns
print(f"MODEL SURFACE  (VERSION='{VERSION}', column='{model_col}')")

model_vol_df = vol_results[['time_to_maturity', 'strike', model_col]].copy()
model_vol_df = model_vol_df.rename(columns={model_col: 'implied_normal_vol'})
model_vol_df = model_vol_df.dropna(subset=['implied_normal_vol'])

mdl_arb_df, mdl_summary = check_market_arbitrage(
    model_vol_df, fwd_key_rate, fwd_ois,
    tau=0.25, h_T=0.1, h_K=0.005,
    plot=True, verbose=True, surface_name='Model',
)